# Fine-tune Qwen2.5-3B-Instruct bằng Unsloth (LoRA) trên VlogQA
Notebook này hướng dẫn fine-tune mô hình **Qwen2.5-3B-Instruct** với **LoRA** qua thư viện **Unsloth** cho tập dữ liệu **VlogQA** (đọc hiểu hội thoại tiếng Việt).

VlogQA là dataset **Extractive QA** (dạng SQuAD): mô hình cần trích xuất một đoạn văn bản ngắn từ context làm câu trả lời.

**Evaluation metrics:** Exact Match (EM) và F1-score.

In [ ]:
# Bước 1: Gỡ cài đặt các gói cũ để tránh xung đột
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [2]:
# Bước 2: Cài đặt lại PyTorch với CUDA 12.8
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 203.3 MB/s  0:00:03a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 197.8 MB/s  0:00:01a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 111.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.3/820.3 MB 230.5 MB/s  0:00:03eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 212.7 MB/s  0:00:00
  Attempting uninstall: cuda-toolkit
    Found existing installation: cuda-toolkit 13.0.3.0
    Uninstalling cuda-toolkit-13.0.3.0:
      Successfully uninstalled cuda-toolkit-13.0.3.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 82.0.1
    Uninstalling setuptools-82.0.1:
      Successfully uninstalled setuptools-82.0.1━━━━━━━━━━━━━━━━━━ 2/7 [setuptools]
  Attempting uninstall: nvidia-nccl-cu12━━━━━━━━━━━━━━━━

In [1]:
# Bước 3: Cài đặt transformers, trl, peft, accelerate, bitsandbytes, xformers và unsloth
!pip install transformers trl peft accelerate bitsandbytes xformers
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 162.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 72.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 204.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 200.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 126.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 56.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 210.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 190.7 MB/s  0:00:020:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 202.1 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 176.0 MB/s  0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 214.2 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 214.4 MB/s  0:00:00 eta 0:00:

## 1. Load Model và Tokenizer với Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 8192  # Tăng từ 1536 → bao phủ P90 VlogQA thực tế (~7052 tokens)
dtype = None           # None để auto-detect (BFloat16 cho 5070 Ti - Blackwell arch)
load_in_4bit = False   # LoRA thuần: nạp model BF16 (~6GB), không quantize

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Tải model thành công!")

## 2. Cấu hình LoRA Adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                          # Rank của LoRA (16 là cân bằng tốt)
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 32,                 # Scaling factor (thường = 2 * r)
    lora_dropout = 0.05,             # Dropout nhẹ để tránh overfitting
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Tiết kiệm VRAM 30%
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("Cấu hình LoRA thành công!")

## 3. Chuẩn bị Dataset VlogQA

In [ ]:
import json
from datasets import Dataset

# ==========================================
# Đọc dữ liệu VlogQA (dạng SQuAD)
# ==========================================
def load_vlogqa(file_path):
    """Đọc file VlogQA và flatten thành danh sách các cặp (context, question, answer)."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                # Lấy câu trả lời đầu tiên (có thể có nhiều annotators)
                if qa["answers"]:
                    answer = qa["answers"][0]["text"]
                else:
                    answer = ""  # Bỏ qua câu không có đáp án
                if answer:  # Chỉ lấy mẫu có đáp án
                    samples.append({
                        "context": context,
                        "question": question,
                        "answer": answer,
                        "id": qa.get("id", ""),
                    })
    return samples

# Đường dẫn dữ liệu (điều chỉnh nếu cần)
TRAIN_PATH = "train.json"
DEV_PATH   = "dev.json"
TEST_PATH  = "test.json"

train_samples = load_vlogqa(TRAIN_PATH)
print(f"Tổng số mẫu train: {len(train_samples)}")
print(f"Ví dụ mẫu đầu tiên:")
print(f"  Context (100 ký tự đầu): {train_samples[0]['context'][:100]}")
print(f"  Question: {train_samples[0]['question']}")
print(f"  Answer: {train_samples[0]['answer']}")

In [ ]:
import json
from datasets import Dataset

# ==========================================
# Đọc dữ liệu VlogQA (dạng SQuAD)
# ==========================================
def load_vlogqa(file_path):
    """Đọc file VlogQA và flatten thành danh sách các cặp (context, question, answer)."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                if qa["answers"]:
                    answer = qa["answers"][0]["text"]
                else:
                    answer = ""
                if answer:
                    samples.append({
                        "context":  context,
                        "question": question,
                        "answer":   answer,
                        "id":       qa.get("id", ""),
                    })
    return samples


# ==========================================
# Anchor-based Context Truncation
# ==========================================
MAX_CONTEXT_TOKENS = 7500

def truncate_context_around_answer(context, answer, tokenizer, max_ctx_tokens=MAX_CONTEXT_TOKENS):
    answer_start = context.find(answer)
    if answer_start == -1:
        ctx_ids = tokenizer.encode(context, add_special_tokens=False)
        if len(ctx_ids) <= max_ctx_tokens:
            return context
        return tokenizer.decode(ctx_ids[:max_ctx_tokens], skip_special_tokens=True)

    before_text = context[:answer_start]
    after_text  = context[answer_start + len(answer):]

    before_ids = tokenizer.encode(before_text, add_special_tokens=False)
    answer_ids = tokenizer.encode(answer,       add_special_tokens=False)
    after_ids  = tokenizer.encode(after_text,   add_special_tokens=False)

    if len(before_ids) + len(answer_ids) + len(after_ids) <= max_ctx_tokens:
        return context

    budget = max_ctx_tokens - len(answer_ids)
    half   = budget // 2

    before_keep = before_ids[-half:]
    after_keep  = after_ids[:budget - half]

    merged_ids = before_keep + answer_ids + after_keep
    return tokenizer.decode(merged_ids, skip_special_tokens=True)


# ==========================================
# System Prompt & User Template (đồng bộ Train/Eval)
# ==========================================
SYSTEM_PROMPT = (
    "Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. "
    "QUY TẮC BẮT BUỘC:\n"
    "1) Chỉ trả về ĐÚNG cụm từ xuất hiện nguyên văn trong đoạn văn, không thêm từ nào khác.\n"
    "2) Không viết câu hoàn chỉnh, không giải thích, không thêm \'Đáp án:\' hay bất kỳ tiền tố nào.\n"
    "3) Câu trả lời phải là một chuỗi ký tự CÓ MẶT trong đoạn văn được cung cấp."
)

USER_PROMPT_TEMPLATE = (
    "Trích xuất câu trả lời từ đoạn văn. Chỉ trả về cụm từ trong đoạn văn.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    "Câu trả lời (span-only):"
)


def format_prompt_train(context, question, answer, tokenizer):
    """Tạo prompt đầy đủ (input + output) để huấn luyện."""
    context_cropped = truncate_context_around_answer(context, answer, tokenizer)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": USER_PROMPT_TEMPLATE.format(
                context=context_cropped,
                question=question
            )
        },
        {"role": "assistant", "content": answer},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def format_prompt_inference(context, question, tokenizer):
    """Tạo prompt inference (chỉ input, không có answer)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": USER_PROMPT_TEMPLATE.format(
                context=context[:7500],
                question=question
            )
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


In [ ]:
# ==========================================
# TẠO DATASET CHO HUẤN LUYỆN VÀ VALIDATION
# ==========================================
from datasets import Dataset

TRAIN_PATH = "train.json"
DEV_PATH   = "dev.json"

# --- Train dataset ---
train_samples = load_vlogqa(TRAIN_PATH)
print(f"Tổng số mẫu train: {len(train_samples)}")

print("\nĐang format prompt cho train dataset...")
train_texts = [
    format_prompt_train(
        context   = s["context"],
        question  = s["question"],
        answer    = s["answer"],
        tokenizer = tokenizer
    )
    for s in train_samples
]
dataset = Dataset.from_dict({"text": train_texts})
print(f"Train dataset: {len(dataset)} mẫu")

# --- Eval dataset (dùng dev.json cho Early Stopping) ---
dev_samples = load_vlogqa(DEV_PATH)
print(f"\nTổng số mẫu dev: {len(dev_samples)}")

print("Đang format prompt cho eval dataset...")
dev_texts = [
    format_prompt_train(
        context   = s["context"],
        question  = s["question"],
        answer    = s["answer"],
        tokenizer = tokenizer
    )
    for s in dev_samples
]
eval_dataset = Dataset.from_dict({"text": dev_texts})
print(f"Eval dataset: {len(eval_dataset)} mẫu")

print(f"\nVí dụ prompt train (100 ký tự đầu):")
print(f"{train_texts[0][:150]}...")


## 4. Cấu hình và Bắt đầu Huấn luyện (Fine-Tuning)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
from unsloth import is_bfloat16_supported
import sys

# Số epoch tối đa (Early Stopping sẽ dừng sớm hơn nếu cần)
MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2   # Dừng nếu eval_loss không giảm sau 2 epoch liên tiếp

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dataset,
    eval_dataset     = eval_dataset,   # ← Thêm validation set để Early Stopping hoạt động
    dataset_text_field = "text",
    max_seq_length   = max_seq_length,
    dataset_num_proc = 1,
    packing          = True,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    args = TrainingArguments(
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 8,       # Effective batch = 16
        warmup_steps                 = 50,
        num_train_epochs             = MAX_EPOCHS,
        learning_rate                = 2e-4,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 20,
        optim                        = "adamw_8bit",
        weight_decay                 = 0.01,
        lr_scheduler_type            = "cosine",
        seed                         = 3407,
        output_dir                   = "outputs_vlogqa",

        # === Cấu hình Evaluation & Early Stopping ===
        eval_strategy                = "epoch",     # Đánh giá sau mỗi epoch
        save_strategy                = "epoch",     # Lưu checkpoint sau mỗi epoch
        load_best_model_at_end       = True,        # Tự động load checkpoint tốt nhất khi kết thúc
        metric_for_best_model        = "eval_loss", # Dùng eval_loss làm tiêu chí
        greater_is_better            = False,       # eval_loss càng nhỏ càng tốt
        save_total_limit             = 2,           # Giữ tối đa 2 checkpoint (tiết kiệm ổ cứng)

        report_to                    = "none",
    ),
)

# Fix sys.modules để tránh lỗi pickle với unsloth
real_config_cls = type(trainer.args)
real_trainer_cls = type(trainer)
sys.modules['trl.trainer.sft_config'] = sys.modules[real_config_cls.__module__]
sys.modules['trl.trainer.sft_trainer'] = sys.modules[real_trainer_cls.__module__]
sys.modules[real_config_cls.__module__].SFTConfig = real_config_cls
sys.modules[real_trainer_cls.__module__].SFTTrainer = real_trainer_cls

# Bắt đầu train!
print(f"\nBắt đầu fine-tuning với Early Stopping (patience={EARLY_STOPPING_PATIENCE}, max {MAX_EPOCHS} epochs)...")
trainer_stats = trainer.train()

# In tóm tắt kết quả
print(f"\nHoàn tất Training!")
print(f"Epochs đã chạy: {trainer_stats.metrics.get('epoch', 'N/A'):.2f}")
print(f"Train Loss cuối: {trainer_stats.metrics.get('train_loss', 'N/A'):.4f}")


## 5. Kiểm tra Inference nhanh sau khi huấn luyện

In [ ]:
# Kích hoạt chế độ sinh token nhanh của Unsloth (2x tốc độ)
FastLanguageModel.for_inference(model)

# Thử nghiệm với mẫu đầu tiên trong tập train
sample = train_samples[0]
prompt = format_prompt_inference(sample["context"], sample["question"], tokenizer)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens = 64,
    use_cache = True,
    do_sample = False,    # Greedy decoding cho QA
    temperature = 1.0,
)

input_length = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_length:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("\n--- KẾT QUẢ INFERENCE ---")
print(f"Câu hỏi : {sample['question']}")
print(f"Đáp án đúng : {sample['answer']}")
print(f"Model trả lời: {response}")

## 6. Lưu Model LoRA

In [ ]:
# Chỉ lưu LoRA weights (nhẹ, vài chục MB)
LORA_SAVE_PATH = "qwen2.5-3b-instruct-lora-vlogqa"

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)

print(f"Đã lưu thành công mô hình LoRA tại: {LORA_SAVE_PATH}")

# Tùy chọn: Merge LoRA vào model gốc và lưu dạng full weights
# model.save_pretrained_merged("qwen2.5-1.5b-instruct-vlogqa-merged", tokenizer, save_method="merged_16bit")

---
## 7. Đánh giá (Evaluation) trên tập Test
### Metrics: Exact Match (EM) và F1-score

- **Exact Match (EM):** Tỷ lệ câu trả lời của model khớp **hoàn toàn** với đáp án (sau khi chuẩn hóa text).
- **F1-score:** Đo lường overlap từ-vựng giữa dự đoán và đáp án (tính theo cấp độ token).

> **Lưu ý:** Cell này có thể chạy trong một session mới sau khi đã train và lưu model.

In [3]:
import json
import re
import string
import unicodedata
import torch
from tqdm.auto import tqdm
from collections import Counter

# ==========================================
# CẤU HÌNH
# ==========================================
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_PATH    = "qwen2.5-3b-instruct-lora-vlogqa"
TEST_DATA_PATH  = "test.json"
MAX_EVAL_TOKENS = 7500

# ==========================================
# ĐỒNG BỘ PROMPT (Giống hệt lúc train)
# ==========================================
SYSTEM_PROMPT = (
    "Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. "
    "QUY TẮC BẮT BUỘC:\n"
    "1) Chỉ trả về ĐÚNG cụm từ xuất hiện nguyên văn trong đoạn văn, không thêm từ nào khác.\n"
    "2) Không viết câu hoàn chỉnh, không giải thích, không thêm \'Đáp án:\' hay bất kỳ tiền tố nào.\n"
    "3) Câu trả lời phải là một chuỗi ký tự CÓ MẶT trong đoạn văn được cung cấp."
)

USER_PROMPT_TEMPLATE = (
    "Trích xuất câu trả lời từ đoạn văn. Chỉ trả về cụm từ trong đoạn văn.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    "Câu trả lời (span-only):"
)

# Regex dọn sạch tiền tố thừa mà model hay sinh ra
PREFIX_RE = re.compile(
    r"^(đáp án|answer|câu trả lời|theo đoạn văn|trong đoạn văn|trả lời|span-only)\s*[:\-]?\s*",
    re.IGNORECASE,
)


/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to load /opt/venv/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /opt/venv/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /opt/venv/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /opt/venv/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


In [ ]:
# ==========================================
# HÀM TIỆN ÍCH: CHUẨN HÓA, METRIC & POST-PROCESSING
# ==========================================

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text or "")
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())


def compute_exact_match(prediction: str, ground_truth: str) -> int:
    return int(normalize_text(prediction) == normalize_text(ground_truth))


def compute_f1_token(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_text(prediction).split()
    true_tokens = normalize_text(ground_truth).split()
    if len(pred_tokens) == 0 and len(true_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(true_tokens) == 0:
        return 0.0
    common = Counter(pred_tokens) & Counter(true_tokens)
    num_common = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall    = num_common / len(true_tokens)
    return (2 * precision * recall) / (precision + recall)


def get_best_score(prediction: str, answers: list) -> tuple:
    """Tính EM và F1 tốt nhất so với tất cả các đáp án (best-of-N)."""
    em_scores = [compute_exact_match(prediction, ans["text"]) for ans in answers]
    f1_scores = [compute_f1_token(prediction, ans["text"]) for ans in answers]
    return max(em_scores), max(f1_scores)


def clean_prediction(raw: str) -> str:
    """Dọn sạch các tiền tố thừa mà model hay sinh ra (vd: 'Đáp án: ...')."""
    pred = raw.strip().split("\n")[0].strip().strip('"\'\' ')
    return PREFIX_RE.sub("", pred).strip()


def find_best_span(prediction: str, context: str) -> str:
    """
    Trượt sliding window qua context để tìm span có token-F1
    cao nhất so với prediction của model.
    Hiệu quả khi model paraphrase nhẹ thay vì copy nguyên văn.
    """
    pred_norm = normalize_text(prediction)
    if not pred_norm:
        return prediction

    pred_words = pred_norm.split()
    ctx_words  = context.split()
    n = len(pred_words)

    if n == 0 or len(ctx_words) == 0:
        return prediction

    best_f1   = compute_f1_token(prediction, context)
    best_span = prediction

    min_w = max(1, n // 2)
    max_w = min(2 * n + 5, len(ctx_words))

    for w in range(min_w, max_w + 1):
        for start in range(len(ctx_words) - w + 1):
            span_orig = " ".join(ctx_words[start:start + w])
            f1 = compute_f1_token(prediction, span_orig)
            if f1 > best_f1:
                best_f1   = f1
                best_span = span_orig

    # Chỉ trả về span mới nếu cải thiện thực sự
    return best_span if best_f1 >= 0.5 else prediction


print("Đã định nghĩa: normalize_text, compute_exact_match, compute_f1_token")
print("               get_best_score, clean_prediction, find_best_span")


In [8]:
# ==========================================
# LOAD MODEL ĐÃ FINE-TUNE (TỐI ƯU TỐC ĐỘ)
# ==========================================
from unsloth import FastLanguageModel
import torch

print("Loading Tokenizer và Model bằng Unsloth FastInference...")
model_eval, tokenizer_eval = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_PATH,  # Load thẳng folder LoRA, Unsloth tự tìm base model
    max_seq_length = 8192,
    dtype = torch.bfloat16,     # BF16 tối ưu cho RTX 40-series
    load_in_4bit = False,
)

# KÍCH HOẠT NATIVE FAST INFERENCE CỦA UNSLOTH (Tăng 2x tốc độ sinh token)
FastLanguageModel.for_inference(model_eval)

print("Load model hoàn tất! (Sử dụng Unsloth FastInference + FlashAttention)")


Loading Tokenizer và Model bằng Unsloth FastInference...
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Ti. Num GPUs = 1. Max memory: 15.576 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!



Unsloth 2026.7.2 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Load model hoàn tất! (Sử dụng Unsloth FastInference + FlashAttention)


In [9]:
# ==========================================
# HÀM TẠO PROMPT INFERENCE (đồng bộ với train)
# ==========================================
def format_prompt_inference_eval(context, question, tokenizer, max_tokens=7500):
    """Tạo prompt cho inference - cùng template với lúc train."""
    ctx_ids = tokenizer.encode(context, add_special_tokens=False)
    if len(ctx_ids) > max_tokens:
        context = tokenizer.decode(ctx_ids[:max_tokens], skip_special_tokens=True)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": USER_PROMPT_TEMPLATE.format(
                context=context,
                question=question
            )
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# ==========================================
# ĐỌC DỮ LIỆU TEST
# ==========================================
def load_vlogqa_with_all_answers(file_path):
    """Đọc VlogQA và giữ nguyên tất cả các đáp án để tính best-of-N metric."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                if qa["answers"]:
                    samples.append({
                        "id": qa.get("id", ""),
                        "context": context,
                        "question": qa["question"],
                        "answers": qa["answers"],
                    })
    return samples

test_samples = load_vlogqa_with_all_answers(TEST_DATA_PATH)
print(f"Tổng số câu hỏi test: {len(test_samples)}")


Tổng số câu hỏi test: 1062
Các hàm metric và best-span đã sẵn sàng!


In [10]:
# ==========================================
# HÀM TẠO PROMPT INFERENCE (đồng bộ với train)
# ==========================================
def format_prompt_inference_eval(context, question, tokenizer, max_tokens=7500):
    """Tạo prompt cho inference - cùng template với lúc train."""
    ctx_ids = tokenizer.encode(context, add_special_tokens=False)
    if len(ctx_ids) > max_tokens:
        context = tokenizer.decode(ctx_ids[:max_tokens], skip_special_tokens=True)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": USER_PROMPT_TEMPLATE.format(
                context=context,
                question=question
            )
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# ==========================================
# ĐỌC DỮ LIỆU TEST
# ==========================================
def load_vlogqa_with_all_answers(file_path):
    """Đọc VlogQA và giữ nguyên tất cả các đáp án để tính best-of-N metric."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                if qa["answers"]:
                    samples.append({
                        "id": qa.get("id", ""),
                        "context": context,
                        "question": qa["question"],
                        "answers": qa["answers"],
                    })
    return samples

test_samples = load_vlogqa_with_all_answers(TEST_DATA_PATH)
print(f"Tổng số câu hỏi test: {len(test_samples)}")


Tổng số câu hỏi test: 1062


In [12]:
# ==========================================
# CHẠY INFERENCE TRÊN TOÀN BỘ TẬP TEST (CÓ MONITOR)
# ==========================================
import time

all_em_raw      = []
all_f1_raw      = []
all_em_span     = []
all_f1_span     = []
all_predictions = []

TOTAL = len(test_samples)
LOG_EVERY = 50           # In kết quả chi tiết mỗi N câu
SUMMARY_EVERY = 100      # In bảng tóm tắt metric mỗi N câu

print(f"Bắt đầu đánh giá trên {TOTAL} câu hỏi...")
print(f"(In chi tiết mỗi {LOG_EVERY} câu | Tóm tắt metric mỗi {SUMMARY_EVERY} câu)\n")
print("=" * 75)

start_time = time.time()
pbar = tqdm(test_samples, desc="Evaluating", total=TOTAL, ncols=80)

for idx, sample in enumerate(pbar, 1):
    t0 = time.time()

    prompt = format_prompt_inference_eval(
        context   = sample["context"],
        question  = sample["question"],
        tokenizer = tokenizer_eval
    )

    inputs = tokenizer_eval(
        prompt,
        return_tensors = "pt",
        truncation = True,
        max_length = 8192,
    ).to(model_eval.device)

    with torch.no_grad():
        outputs = model_eval.generate(
            **inputs,
            max_new_tokens = 64,
            do_sample = False,
            temperature = 1.0,
            pad_token_id = tokenizer_eval.pad_token_id,
            eos_token_id = tokenizer_eval.eos_token_id,
        )

    input_length   = inputs["input_ids"].shape[1]
    generated_ids  = outputs[0][input_length:]
    raw_prediction = tokenizer_eval.decode(generated_ids, skip_special_tokens=True).strip()

    # Post-processing
    cleaned_prediction = clean_prediction(raw_prediction)
    span_prediction    = find_best_span(cleaned_prediction, sample["context"])

    # Tính metric
    em_raw,  f1_raw  = get_best_score(cleaned_prediction, sample["answers"])
    em_span, f1_span = get_best_score(span_prediction,    sample["answers"])

    all_em_raw.append(em_raw);   all_f1_raw.append(f1_raw)
    all_em_span.append(em_span); all_f1_span.append(f1_span)
    all_predictions.append({
        "id":               sample["id"],
        "question":         sample["question"],
        "ground_truth":     sample["answers"][0]["text"],
        "raw_prediction":   raw_prediction,
        "clean_prediction": cleaned_prediction,
        "span_prediction":  span_prediction,
        "em_raw":  em_raw,  "f1_raw":  f1_raw,
        "em_span": em_span, "f1_span": f1_span,
    })

    step_time = time.time() - t0
    elapsed   = time.time() - start_time
    eta       = (elapsed / idx) * (TOTAL - idx)

    # Cập nhật thanh tiến trình
    cur_em  = sum(all_em_span) / idx * 100
    cur_f1  = sum(all_f1_span) / idx * 100
    pbar.set_postfix({
        "EM": f"{cur_em:.1f}%",
        "F1": f"{cur_f1:.1f}%",
        "ETA": f"{eta/60:.1f}m"
    })

    # In chi tiết mỗi LOG_EVERY câu
    if idx % LOG_EVERY == 0 or idx == 1:
        status = "✓" if em_span == 1 else ("~" if f1_span >= 0.5 else "✗")
        tqdm.write(
            f"[{idx:>4}/{TOTAL}] {status} "
            f"Q: {sample['question'][:45]:<45} | "
            f"GT: {sample['answers'][0]['text'][:20]:<20} | "
            f"Pred: {span_prediction[:20]:<20} | "
            f"F1={f1_span:.2f}"
        )

    # In tóm tắt metric mỗi SUMMARY_EVERY câu
    if idx % SUMMARY_EVERY == 0:
        tqdm.write("\n" + "-" * 55)
        tqdm.write(f"  [Checkpoint {idx}/{TOTAL}] Đã chạy {elapsed:.0f}s | ETA ~{eta/60:.1f} phút")
        tqdm.write(f"  EM  (raw):  {sum(all_em_raw)/idx*100:.2f}%  |  EM  (span): {sum(all_em_span)/idx*100:.2f}%")
        tqdm.write(f"  F1  (raw):  {sum(all_f1_raw)/idx*100:.2f}%  |  F1  (span): {sum(all_f1_span)/idx*100:.2f}%")
        tqdm.write("-" * 55 + "\n")

total_time = time.time() - start_time
print(f"\nHoàn tất! Tổng thời gian: {total_time/60:.1f} phút")



Bắt đầu đánh giá trên 1062 câu hỏi...



Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=64) 

[1/1062] Q: Lò được làm nóng trước khi nướng bao lâu... | Truth: 10 phút | Raw: 10 phút | Span: 10 phút | EM_raw=1 EM_span=1 F1_raw=1.000 F1_span=1.000


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/1062] Q: Đầu bếp sử dụng gì để trang trí bánh?... | Truth: mấy cái hoa hồi | Raw: hoa hồi | Span: hoa hồi | EM_raw=0 EM_span=0 F1_raw=0.667 F1_span=0.667


...

[1059/1062] Q: Đánh bột thế nào để không bị óc trâu?... | Truth: quay một chiều | Raw: khi mở ra списка xung quanh U  | Span: à ừ ừ i fell tiếp tục I đi bộ  | EM_raw=0 EM_span=0 F1_raw=0.044 F1_span=0.000


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1060/1062] Q: Món bánh trứng phù hợp ăn vào buổi nào?... | Truth: sáng | Raw: bữa sáng | Span: bữa sáng | EM_raw=0 EM_span=0 F1_raw=0.667 F1_span=0.667


Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1061/1062] Q: Sử dụng loại trứng gì để làm bánh?... | Truth: trứng gà | Raw: Mình bấm Trứng lớn | Span: trứng lớn Mình | EM_raw=0 EM_span=0 F1_raw=0.333 F1_span=0.400
[1062/1062] Q: Món bánh trứng thường ăn chung với loại ... | Truth: tương ớt hoặc là sốt Mayonnais | Raw: xốt Mayonnaise | Span: Mayonnaise | EM_raw=0 EM_span=0 F1_raw=0.250 F1_span=0.286

Hoàn tất inference!


In [13]:
# ==========================================
# IN KẾT QUẢ ĐÁNH GIÁ
# ==========================================
total = len(all_em_raw)

em_raw_score   = sum(all_em_raw)  / total * 100
f1_raw_score   = sum(all_f1_raw)  / total * 100
em_span_score  = sum(all_em_span) / total * 100
f1_span_score  = sum(all_f1_span) / total * 100

print("\n" + "=" * 60)
print("   KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST - VlogQA")
print("   Model: Qwen2.5-3B-Instruct + LoRA (Unsloth)")
print("=" * 60)
print(f"  Tổng số câu hỏi test: {total}")
print()
print(f"  {'Metric':<25} {'Raw (no post-proc)':>20} {'Best-Span':>20}")
print(f"  {'-'*65}")
print(f"  {'Exact Match (EM)':<25} {em_raw_score:>19.2f}% {em_span_score:>19.2f}%")
print(f"  {'F1-score (token)':<25} {f1_raw_score:>19.2f}% {f1_span_score:>19.2f}%")
print("=" * 60)

# Phân phối F1 (span version)
f1_buckets = {"0-0.2": 0, "0.2-0.5": 0, "0.5-0.8": 0, "0.8-1.0": 0, "1.0": 0}
for f1 in all_f1_span:
    if f1 == 1.0:    f1_buckets["1.0"] += 1
    elif f1 >= 0.8:  f1_buckets["0.8-1.0"] += 1
    elif f1 >= 0.5:  f1_buckets["0.5-0.8"] += 1
    elif f1 >= 0.2:  f1_buckets["0.2-0.5"] += 1
    else:            f1_buckets["0-0.2"] += 1

print(f"\n  EM đúng (span): {sum(all_em_span)}/{total} câu")
print("\n  Phân phối F1-score (Best-Span):")
for bucket, count in f1_buckets.items():
    print(f"    F1 = {bucket}: {count} câu ({count/total*100:.1f}%)")

# Lưu kết quả
with open("eval_results_vlogqa_3b.json", "w", encoding="utf-8") as f:
    json.dump({
        "em_raw":    round(em_raw_score,  4),
        "f1_raw":    round(f1_raw_score,  4),
        "em_span":   round(em_span_score, 4),
        "f1_span":   round(f1_span_score, 4),
        "total":     total,
        "predictions": all_predictions
    }, f, ensure_ascii=False, indent=2)

print("\nKết quả chi tiết đã được lưu tại: eval_results_vlogqa_3b.json")


   KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST - VlogQA
   Model: Qwen2.5-3B-Instruct + LoRA (Unsloth)
  Tổng số câu hỏi test: 1062

  Metric                      Raw (no post-proc)            Best-Span
  -----------------------------------------------------------------
  Exact Match (EM)                        21.85%               20.62%
  F1-score (token)                        41.10%               38.68%

  EM đúng (span): 219/1062 câu

  Phân phối F1-score (Best-Span):
    F1 = 0-0.2: 498 câu (46.9%)
    F1 = 0.2-0.5: 141 câu (13.3%)
    F1 = 0.5-0.8: 160 câu (15.1%)
    F1 = 0.8-1.0: 44 câu (4.1%)
    F1 = 1.0: 219 câu (20.6%)

Kết quả chi tiết đã được lưu tại: eval_results_vlogqa_3b.json


In [ ]:
# ==========================================
# XEM CÁC MẪU SAI ĐỂ PHÂN TÍCH
# ==========================================
wrong_preds = [p for p in all_predictions if p["em"] == 0]
print(f"\nSố câu EM = 0 (sai hoàn toàn): {len(wrong_preds)}")
print("\n--- 5 mẫu sai đầu tiên ---")
for p in wrong_preds[:5]:
    print(f"\n  Q: {p['question']}")
    print(f"  Ground truth: {p['ground_truth']}")
    print(f"  Prediction  : {p['prediction']}")
    print(f"  F1          : {p['f1']:.4f}")

# Lưu kết quả ra file JSON để phân tích sau
with open("eval_results_vlogqa_3b.json", "w", encoding="utf-8") as f:
    json.dump({
        "exact_match": round(exact_match_score, 4),
        "f1_score": round(f1_score_avg, 4),
        "total": total,
        "predictions": all_predictions
    }, f, ensure_ascii=False, indent=2)

print("\nKết quả chi tiết đã được lưu tại: eval_results_vlogqa_3b.json")